In [4]:
import os
import json
import pandas as pd
from typing import List, Dict, Union
from pydantic import BaseModel, Field
from groq import Groq
from google.colab import userdata

# ==========================================
# 1. SECURE INITIALIZATION & SETTINGS
# ==========================================
# Securely pulls the key from Colab Secrets instead of hardcoding
try:
    api_key = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=api_key)
except Exception:
    raise ValueError("Please add your 'GROQ_API_KEY' to the Colab Secrets (🔑) panel and enable access.")

MODEL = "llama-3.3-70b-versatile"

df = pd.read_csv("/content/patient_data (1).csv")

# Ensure column types are set for clean lookups
df['patient_id'] = df['patient_id'].astype(str).str.strip()
df['vitals_bp_systolic'] = pd.to_numeric(df['vitals_bp_systolic'], errors='coerce').fillna(0).astype(int)
df['vitals_spo2'] = pd.to_numeric(df['vitals_spo2'], errors='coerce').fillna(100).astype(int)
df['days_since_last_visit'] = pd.to_numeric(df['days_since_last_visit'], errors='coerce').fillna(0).astype(int)
df['missed_last_appointment'] = df['missed_last_appointment'].astype(str).str.strip()

# ==========================================
# 2. PYDANTIC SCHEMAS (Data Safety Guardrails)
# ==========================================
class PatientDataSchema(BaseModel):
    patient_id: str
    patient_name: str
    vitals_bp_systolic: int = Field(..., description="Systolic blood pressure value")
    vitals_spo2: int = Field(..., description="Blood oxygen saturation percentage")
    days_since_last_visit: int
    missed_last_appointment: str

class RiskListSchema(BaseModel):
    risks: List[str] = Field(..., description="List of identified clinical risks")

# ==========================================
# 3. PRODUCTION TOOLS
# ==========================================
def get_patient(patient_id: str) -> Dict:
    pid = str(patient_id).strip()
    match = df[df['patient_id'] == pid]
    if match.empty:
        return {"error": f"Patient ID {pid} not found."}

    # Validate structure using Pydantic before allowing downstream processing
    record = match.to_dict(orient='records')[0]
    validated_patient = PatientDataSchema(**record)
    return validated_patient.model_dump()

def risk_scoring(patient_data: Dict) -> Dict:
    # Ensure raw input conforms to structure
    if isinstance(patient_data, str):
        patient_data = json.loads(patient_data)

    risks = []
    if patient_data.get('vitals_bp_systolic', 0) > 140:
        risks.append("High Blood Pressure")
    if patient_data.get('vitals_spo2', 100) < 92:
        risks.append("Low Oxygen")
    if patient_data.get('days_since_last_visit', 0) > 30:
        risks.append("Long Gap Since Visit")
    if str(patient_data.get('missed_last_appointment', '')).lower() == "yes":
        risks.append("Missed Appointment")

    return {"risks": risks}

def generate_actions(risks_data: Dict) -> List[str]:
    risks = risks_data.get('risks', [])
    actions = []
    if "High Blood Pressure" in risks:
        actions.append("Schedule urgent BP review")
    if "Low Oxygen" in risks:
        actions.append("Immediate respiratory assessment")
    if "Missed Appointment" in risks:
        actions.append("Call patient for follow-up")
    if "Long Gap Since Visit" in risks:
        actions.append("Schedule routine checkup")
    return actions

# Mapping table for the local runtime
TOOL_MAP = {
    "get_patient": get_patient,
    "risk_scoring": risk_scoring,
    "generate_actions": generate_actions
}

# ==========================================
# 4. NATIVE API TOOL DEFINITIONS (Function Calling)
# ==========================================
GROQ_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_patient",
            "description": "Retrieves comprehensive clinical and profile information for a specific patient identifier.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_id": {"type": "string", "description": "The unique patient ID string (e.g. 'P0018')"}
                },
                "required": ["patient_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "risk_scoring",
            "description": "Evaluates vitals and timelines to compile a structured list of medical and operational risks.",
            "parameters": {
                "type": "object",
                "properties": {
                    "patient_data": {
                        "type": "object",
                        "description": "The complete patient payload returned from get_patient tool."
                    }
                },
                "required": ["patient_data"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_actions",
            "description": "Builds a definitive operational protocol or clinical action step array based on a provided risks tracking dictionary.",
            "parameters": {
                "type": "object",
                "properties": {
                    "risks_data": {
                        "type": "object",
                        "description": "A dictionary object containing the list of verified risks under the 'risks' key."
                    }
                },
                "required": ["risks_data"]
            }
        }
    }
]

# ==========================================
# 5. AGENT ENGINE
# ==========================================
SYSTEM_PROMPT = """You are an advanced Clinical AI Orchestrator.
Your objective is to systematically coordinate patient analysis.

Always progress through these steps:
1. Call get_patient to access their current record.
2. Submit that data payload into risk_scoring to extract clinical issues.
3. Pass those concerns to generate_actions to create your final tracking protocol.

Once actions are built, provide your final response to the user as a clean, definitive list of recommendations. Do not describe your tool steps in the final answer."""

def run_production_agent(patient_id: str) -> List[str]:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Execute clinical follow-up planning for patient ID: {patient_id}"}
    ]

    # Give it up to 8 conversational back-and-forth loops to resolve dependencies
    for _ in range(8):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=GROQ_TOOLS,
            tool_choice="auto",
            temperature=0
        )

        response_message = response.choices[0].message
        messages.append(response_message)

        # Check if the LLM is choosing to invoke a tool natively
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                # Dynamic function routing
                if function_name in TOOL_MAP:
                    # Unwrap named arguments cleanly matching functions
                    arg_value = list(function_args.values())[0]
                    tool_output = TOOL_MAP[function_name](arg_value)
                else:
                    tool_output = f"Error: Tool '{function_name}' is unavailable."

                # Append tool results natively using the specific tool_call_id Groq requested
                messages.append({
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": function_name,
                    "content": json.dumps(tool_output)
                })
        else:
            # If the model emits regular text instead of a tool request, it has found its final answer
            return response_message.content

    return ["Error: Agent loop limit reached without resolution."]

# ==========================================
# 6. BATCH PRODUCTION EXECUTION
# ==========================================
print("\n--- Running Production Pipeline ---")
missed_appointments = df[df['missed_last_appointment'] == "Yes"]
production_results = {}

for pid in missed_appointments['patient_id'].head(5):
    print(f"Processing evaluation for: {pid}...")
    production_results[pid] = run_production_agent(pid)

print("\n--- Final Consolidated Output ---")
for pid, actions in production_results.items():
    print(f"\nPatient {pid} Action Protocol:\n{actions}")


--- Running Production Pipeline ---
Processing evaluation for: P0018...
Processing evaluation for: P0031...
Processing evaluation for: P0036...
Processing evaluation for: P0041...
Processing evaluation for: P0043...

--- Final Consolidated Output ---

Patient P0018 Action Protocol:
Based on the patient data and risk assessment, here is a definitive operational protocol or clinical action step array:

1. Monitor blood pressure regularly to manage hypertension.
2. Schedule a follow-up appointment to review the patient's condition and adjust treatment plans as necessary.
3. Educate the patient on the importance of attending scheduled appointments and provide support to improve adherence.
4. Consider referring the patient to a specialist for further evaluation and management of their condition.

Patient P0031 Action Protocol:
Based on the patient's current record and risk assessment, the following actions are recommended:

* Schedule a follow-up appointment with the patient to monitor the

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')